## Among papers with funding information, distinguish papers with complete country information (every acknowledged funding agency has country information) and papers with incomplete country information.  

In [3]:
import pandas as pd

In [1]:
fund_path="../../data/dropbox/LaborDivision/data/rawdata/article_funding.txt"
funder_cntry_path="../../data/dropbox/LaborDivision/data/additionaldata/funder_country_new.csv"
pub_author_path = "../../data/dropbox/LaborDivision/data/deriveddata/all/pub_author.csv"
cntry_path = "../../data/dropbox/LaborDivision/data/additionaldata/cntry_meta.csv"

In [31]:
pub_funding_path="../../data/dropbox/LaborDivision/data/deriveddata/all/pub_funding.pkl"

In [4]:
fund_df=pd.read_csv(fund_path,sep="|",on_bad_lines='skip')
fund_df.head()

,OST_BK,Agency
0,89364646,Funding Program for Next Generation World-Lead...
1,89364646,JSPS
2,89364646,"Ministry of Education, Culture, Sports, Scienc..."
3,89364646,Japan Society for the Promotion of Science
4,89364646,Japan Society for the Promotion of Science (JSPS)


In [5]:
fund_df.rename(columns={'OST_BK':'ost_bk','Agency':'agency'}, inplace=True)

In [6]:
funder_cntry=pd.read_csv(funder_cntry_path)[['label','agency','cntry']]
funder_cntry.head()

,label,agency,cntry
0,p1,National Natural Science Foundation of China,China
1,p1,National Institutes of Health,United States
2,p1,NIH,United States
3,p1,National Science Foundation,United States
4,p1,NSF,United States


In [7]:
fund_df=fund_df.merge(funder_cntry[['agency','cntry']],on='agency',how='left')

In [8]:
fund_df.head(2)

,ost_bk,agency,cntry
0,89364646,Funding Program for Next Generation World-Lead...,Japan
1,89364646,JSPS,Japan


### articles that at least one funding institution miss the country information

In [11]:
pub_miss_fund=fund_df[fund_df.cntry.isna()]['ost_bk'].unique()
pub_miss_fund = fund_df[fund_df.ost_bk.isin(pub_miss_fund)]
pub_miss_fund = pub_miss_fund[['ost_bk']].drop_duplicates()
pub_miss_fund.loc[:,"funders"]='incomplete'

In [12]:
pub_miss_fund.head()

,ost_bk,funders
26,89364679,incomplete
45,89364717,incomplete
53,89364723,incomplete
55,89364726,incomplete
73,89364733,incomplete


### articles that every funding institution has country information

In [17]:
pub_with_fund=fund_df[~fund_df.ost_bk.isin(pub_miss_fund.ost_bk.unique())] 

In [18]:
cntry_code=pd.read_csv(cntry_path)
df1 = cntry_code[['WoS','Code']].drop_duplicates().rename(columns={'WoS':'cntry'})
df2 = cntry_code[['cntry','Code']].drop_duplicates()
cntry_code = pd.concat([df1,df2])
cntry_code = cntry_code.drop_duplicates()
cntry_code.loc[len(cntry_code)] = ['EU','EU']

In [19]:
pub_with_fund.loc[:,'cntry'] = pub_with_fund['cntry'].apply(lambda x:x.strip())
pub_with_fund=pub_with_fund.merge(cntry_code,on='cntry',how='left')

In [20]:
#remove pubs that without country code
invalid_ids = pub_with_fund[pub_with_fund.Code.isna()].ost_bk.unique()
pub_with_fund=pub_with_fund[~pub_with_fund.ost_bk.isin(invalid_ids)]

In [21]:
len(pub_with_fund.ost_bk.unique())/len(fund_df.ost_bk.unique())

0.576963058714921

In [24]:
pub_with_fund.head()

ost_bk
29768167    [GBR]
29768198    [IRN]
29768203    [USA]
29768231    [USA]
29768270    [USA]
Name: Code, dtype: object

In [23]:
pub_with_fund=pub_with_fund[['ost_bk','Code']].drop_duplicates()
pub_with_fund=pub_with_fund.groupby('ost_bk')['Code'].apply(list)

In [25]:
pub_with_fund = pub_with_fund.reset_index(name='funders')
pub_with_fund.head(2)

,ost_bk,funders
0,29768167,[GBR]
1,29768198,[IRN]


In [26]:
pub_fund = pd.concat([pub_with_fund, pub_miss_fund], ignore_index=True)

In [27]:
pub_fund.head()

,ost_bk,funders
0,29768167,[GBR]
1,29768198,[IRN]
2,29768203,[USA]
3,29768231,[USA]
4,29768270,[USA]


In [32]:
pub_fund.to_pickle(pub_funding_path)

In [40]:
pub_author_path='../../data/dropbox/LaborDivision/data/deriveddata/intl_noalpha/pub_author.csv'
test = pd.read_csv(pub_author_path)

In [43]:
a = test[['ost_bk']].drop_duplicates()

In [44]:
a = a.merge(pub_with_fund, on='ost_bk')

In [45]:
len(a.ost_bk.unique())

1195140

In [41]:
len(test.ost_bk.unique())

3195989

In [42]:
1195140/3195989

0.37394997291918086